# 25. 사업보고서 원문 추출 비교

**목적:** 공시 목록의 `rcept_no`로 실제 사업보고서 원문을 받은 뒤, 단순 평탄화(A)와 제목·표 경계를 표시하는 추출(B)이 검색 가능한 텍스트를 어떻게 만드는지 확인합니다. 원문 구조를 모른 채 chunking·embedding을 비교하면 검색 실패의 원인을 구분하기 어렵습니다.

**범위:** 삼성전자·리노공업의 2024 사업연도 사업보고서 최종 제출본 각 1건. 이는 *2025년 접수일*에 제출된 2024 사업연도 보고서입니다. 다른 기업·결산월까지 일반화하지 않습니다.

**이번에 판단하지 않는 것:** 파서 최종 채택, chunking·embedding·retrieval 성능, LLM 답변 품질. 정답 근거 문단을 독립적으로 표시하기 전에는 Recall@K·MRR을 계산하지 않습니다. API 키와 원문 전체를 출력하지 않습니다.

참고: [OpenDART 공시검색](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS001&apiId=2019001), [공시서류원본파일](https://opendart.fss.or.kr/guide/detail.do?apiGrpCd=DS001&apiId=2019003), [LlamaIndex Node Parser](https://developers.llamaindex.ai/python/framework/module_guides/loading/node_parsers/).

## 1. 환경과 입력 범위

In [ ]:
from collections import Counter
from io import BytesIO
from pathlib import Path
from time import perf_counter
import os
import re
import sys
import zipfile

from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from dotenv import load_dotenv
import pandas as pd
import requests

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / '.env').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 최상위 .env를 찾지 못했습니다.')
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env에 OPENDART_API_KEY가 필요합니다.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from dart.client import find_corp_info
print('실행 환경과 API 키 존재 여부 확인 완료 (키 값은 출력하지 않음)')

In [ ]:
COMPANIES = ['삼성전자', '리노공업']
BUSINESS_YEAR = 2024
RECEIPT_START, RECEIPT_END = '20250101', '20251231'
LIST_URL = 'https://opendart.fss.or.kr/api/list.json'
DOCUMENT_URL = 'https://opendart.fss.or.kr/api/document.xml'
print('회사:', COMPANIES, '| 사업연도:', BUSINESS_YEAR, '| 접수일:', RECEIPT_START, '~', RECEIPT_END)

## 2. 사업보고서 접수번호 선택

`pblntf_detail_ty=A001`은 사업보고서 유형입니다. 목록 결과에서 사업연도 표기까지 확인하고, 여러 최종 제출본이 있으면 이 실험에서는 가장 최근 접수번호를 사용합니다. 선택 후보를 먼저 출력하므로 잘못된 문서를 조용히 고르지 않습니다.

In [ ]:
def get_json(url, params):
    try:
        response = requests.get(url, params=params, timeout=40)
        if response.status_code != 200:
            raise RuntimeError(f'OpenDART HTTP {response.status_code}')
        return response.json()
    except requests.RequestException:
        raise RuntimeError('OpenDART 요청 실패: 연결 또는 시간 제한을 확인하세요.') from None

def list_annual_reports(corp_code):
    params = {'crtfc_key': API_KEY, 'corp_code': corp_code, 'bgn_de': RECEIPT_START,
              'end_de': RECEIPT_END, 'pblntf_detail_ty': 'A001',
              'pblntf_ty': 'A', 'last_reprt_at': 'Y', 'page_count': 100}
    rows = []
    page_no = 1
    expected_total = expected_pages = None
    while True:
        payload = get_json(LIST_URL, {**params, 'page_no': page_no})
        if page_no == 1 and payload.get('status') == '013':
            return []
        if payload.get('status') != '000':
            raise RuntimeError(f"공시 목록 오류: {payload.get('status')} / {payload.get('message')}")
        try:
            total, pages, current = (int(payload[key]) for key in ('total_count', 'total_page', 'page_no'))
        except (KeyError, TypeError, ValueError):
            raise RuntimeError('공시 목록 페이지 메타데이터가 비정상입니다.') from None
        page_rows = payload.get('list')
        if pages < 1 or current != page_no or not isinstance(page_rows, list) or any(not isinstance(row, dict) for row in page_rows):
            raise RuntimeError('공시 목록 페이지 번호 또는 list 형식이 비정상입니다.')
        if expected_total is None:
            expected_total, expected_pages = total, pages
        elif (total, pages) != (expected_total, expected_pages):
            raise RuntimeError('공시 목록 조회 도중 전체 건수 또는 페이지 수가 변경됐습니다.')
        rows.extend(page_rows)
        if page_no >= expected_pages:
            receipts = [str(row.get('rcept_no') or '') for row in rows]
            if any(len(number) != 14 or not number.isdigit() for number in receipts):
                raise RuntimeError('공시 목록에 14자리 숫자가 아닌 접수번호가 있습니다.')
            if len(rows) != expected_total or len(set(receipts)) != len(rows):
                raise RuntimeError('공시 목록의 수집 건수·접수번호가 API 총수와 다릅니다.')
            return rows
        page_no += 1

In [ ]:
report_candidates = []
for company in COMPANIES:
    corp_code = find_corp_info(company, API_KEY)['corp_code']
    for row in list_annual_reports(corp_code):
        if re.search(rf'\b{BUSINESS_YEAR}\.12\b', row.get('report_nm', '')):
            report_candidates.append({'company': company, 'corp_code': corp_code,
                                      'rcept_no': row['rcept_no'], 'rcept_dt': row['rcept_dt'],
                                      'report_nm': row['report_nm']})
if not report_candidates:
    raise RuntimeError('2024.12 사업보고서 후보가 없습니다. 날짜와 보고서명을 직접 확인하세요.')
display(pd.DataFrame(report_candidates).sort_values(['company', 'rcept_no']))

In [ ]:
selected_reports = []
for company in COMPANIES:
    matches = [row for row in report_candidates if row['company'] == company]
    if not matches:
        raise RuntimeError(f'{company}: 해당 사업연도 보고서 후보가 없습니다.')
    selected_reports.append(max(matches, key=lambda row: row['rcept_no']))
display(pd.DataFrame(selected_reports))
print('위 접수번호가 의도한 사업보고서인지 확인한 뒤 다음 셀을 실행하세요.')

## 3. 원문 ZIP 확인

공시검색은 문서 **목록**만 반환합니다. 아래에서 선택한 `rcept_no`로 원문 ZIP을 별도 호출합니다. ZIP 안 XML 파일명과 크기를 출력해 파싱 입력을 확인합니다. 원문 본문 전체는 출력하거나 로컬에 저장하지 않습니다.

In [ ]:
def download_document_xml(rcept_no):
    start = perf_counter()
    try:
        with requests.get(DOCUMENT_URL, params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                          timeout=120, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f'원문 API HTTP {response.status_code}')
            chunks, received = [], 0
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                received += len(chunk)
                if received > 60_000_000:
                    raise RuntimeError('원문 ZIP이 60MB를 넘어 다운로드를 중단합니다.')
                chunks.append(chunk)
    except requests.RequestException:
        raise RuntimeError('원문 API 요청 실패: 연결 또는 시간 제한을 확인하세요.') from None
    payload = b''.join(chunks)
    if not zipfile.is_zipfile(BytesIO(payload)):
        raise RuntimeError(f'{rcept_no}: 원문 응답이 ZIP이 아닙니다. API 상태와 접수번호를 확인하세요.')
    with zipfile.ZipFile(BytesIO(payload)) as archive:
        xml_entries = [info for info in archive.infolist() if info.filename.lower().endswith('.xml')]
        if not xml_entries:
            raise RuntimeError(f'{rcept_no}: ZIP 안에 XML 파일이 없습니다.')
        chosen = max(xml_entries, key=lambda info: info.file_size)
        if chosen.file_size > 60_000_000 or sum(info.file_size for info in archive.infolist()) > 120_000_000:
            raise RuntimeError(f'{rcept_no}: 압축 해제 크기가 실험 한도를 넘습니다.')
        xml_bytes = archive.read(chosen)
        entries = [{'filename': info.filename, 'bytes': info.file_size}
                   for info in archive.infolist() if not info.is_dir()]
    return xml_bytes, chosen.filename, entries, perf_counter() - start

## 4. 원문 구조와 A/B 추출 비교

A는 XML의 텍스트를 한 줄로 이어 붙입니다. B는 같은 텍스트를 사용하되 제목과 표 셀의 경계를 표시합니다. 두 방식 모두 원문의 정확한 문단 구조를 보장하지 않으므로, 출력 예시를 직접 읽고 손실·중복·잡음 여부를 확인합니다.

In [ ]:
def inspect_xml(xml_bytes):
    soup = BeautifulSoup(xml_bytes, 'lxml-xml')
    if soup.find() is None:
        raise RuntimeError('XML에서 태그를 찾지 못했습니다.')
    tag_counts = Counter(tag.name.upper() for tag in soup.find_all(True))
    title_samples = [tag.get_text(' ', strip=True)[:160] for tag in soup.find_all(['TITLE', 'SUBTITLE'])]
    title_samples = [title for title in title_samples if title]
    return soup, tag_counts, title_samples

In [ ]:
def extract_a(soup):
    return re.sub(r'\s+', ' ', soup.get_text(' ', strip=True)).strip()

def extract_b(soup):
    parts = []
    for node in soup.descendants:
        if isinstance(node, Tag) and node.name.upper() == 'TR':
            parts.append('\n')
        if not isinstance(node, NavigableString) or isinstance(node, Comment):
            continue
        value = re.sub(r'\s+', ' ', str(node)).strip()
        if not value:
            continue
        ancestors = {tag.name.upper() for tag in node.parents if tag.name}
        if ancestors & {'TITLE', 'SUBTITLE'}:
            parts.append('\n## ' + value + '\n')
        elif ancestors & {'TD', 'TH'}:
            parts.append(value + ' | ')
        else:
            parts.append(value + ' ')
    return re.sub(r' *\n *', '\n', ''.join(parts)).strip()

In [ ]:
documents, zip_rows, extraction_rows = {}, [], []
for report in selected_reports:
    company = report['company']
    xml_bytes, filename, entries, seconds = download_document_xml(report['rcept_no'])
    soup, counts, titles = inspect_xml(xml_bytes)
    document = {**report, 'xml_filename': filename}
    zip_rows.append({'company': company, 'rcept_no': report['rcept_no'],
                     'zip_entries': len(entries), 'chosen_xml': filename,
                     'xml_bytes': len(xml_bytes), 'download_seconds': round(seconds, 2)})
    print(company, '| ZIP 파일:', entries[:10], '...' if len(entries) > 10 else '')
    print(company, '| 상위 태그:', counts.most_common(12), '| 제목 예시:', titles[:12])
    if not titles:
        print('주의: TITLE/SUBTITLE 태그를 찾지 못했습니다. 원문 구조를 직접 확인하세요.')
    for variant, extractor in [('A_flat', extract_a), ('B_boundaries', extract_b)]:
        start = perf_counter()
        value = extractor(soup)
        if not value:
            raise RuntimeError(f'{company} {variant}: 추출된 텍스트가 없습니다.')
        if variant == 'B_boundaries' and '## ' not in value:
            print(f'주의: {company} B에서 제목 경계가 표시되지 않았습니다.')
        document[variant] = value
        extraction_rows.append({'company': company, 'variant': variant,
                                'characters': len(value), 'lines': value.count('\n') + 1,
                                'marked_headings': value.count('\n## '),
                                'extract_seconds': round(perf_counter() - start, 2)})
    documents[company] = document
    del soup, xml_bytes
display(pd.DataFrame(zip_rows))
display(pd.DataFrame(extraction_rows))

In [ ]:
def context_window(text, keyword, radius=250):
    position = text.find(keyword)
    if position < 0:
        return '키워드를 추출 텍스트에서 찾지 못함'
    return text[max(0, position - radius):position + len(keyword) + radius]

for company, document in documents.items():
    print('\n###', company, '| 접수번호', document['rcept_no'])
    for keyword in ['사업의 내용', '위험', '주요 제품']:
        print('\n키워드:', keyword)
        for variant in ['A_flat', 'B_boundaries']:
            print(variant, '=>', context_window(document[variant], keyword))

## 해석할 때 볼 것

1. 선택된 접수번호가 정말 2024 사업연도 사업보고서인지 확인합니다. `rcept_dt`는 접수일이지 사업연도가 아닙니다.
2. ZIP에 XML이 여러 개라면 가장 큰 XML을 고른 현재 **실험용 규칙**이 본문을 가리키는지 확인합니다. 아닐 경우 여기서 중단하고 문서 선택 규칙을 다시 정해야 합니다.
3. A/B 출력에서 제목, 본문, 표 수치가 누락·중복·깨짐 없이 읽히는지 확인합니다. `characters`나 제목 개수가 많다고 검색 품질이 좋다는 뜻은 아닙니다.
4. 다음 평가에서는 사람이 확인한 원문 근거 문단과 질문을 정답으로 고정한 뒤, 파싱·chunking·retrieval 후보의 Recall@K, MRR, nDCG@K를 비교합니다. 이번 노트북만으로 파서를 채택하지 않습니다.